In [1]:
!pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 7.8 MB/s eta 0:00:00


In [2]:
# ── CELL 2: Mount Drive First ─────────────────────────────────────────────────
# Drive must be mounted before logging is configured because the log file
# is written to Drive. If Drive is not mounted first, the log file handler
# will fail on startup.
from google.colab import drive
drive.mount('/content/drive')

import os

fraud_folder = "/content/drive/MyDrive/fraud_data/"

# Create the folder if it does not exist yet
os.makedirs(fraud_folder, exist_ok=True)

print(f"Drive mounted. Folder ready: {fraud_folder}")
print(f"Current contents: {os.listdir(fraud_folder)}")


Mounted at /content/drive
Drive mounted. Folder ready: /content/drive/MyDrive/fraud_data/
Current contents: ['archive (3).zip', 'train_transaction.csv.zip', 'train_identity.csv.zip', 'train_transaction.csv', 'train_identity.csv', 'PS_20174392719_1491204439457_log.csv', 'insertion_log.txt', 'insertion_log.gdoc']


In [3]:
# ── CELL 3: Imports and Logging ────────────────────────────────────────────────
# Now that Drive is mounted, the log file handler can write to it safely
from pymongo import MongoClient, InsertOne
from pymongo.server_api import ServerApi
from google.colab import userdata
import pandas as pd
import math
import logging
import gc
import zipfile

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(levelname)s — %(message)s",
    handlers=[
        # Permanent log file in Drive — survives after the session closes
        logging.FileHandler("/content/drive/MyDrive/fraud_data/insertion_log.txt"),
        # Also print to console to watch progress in real time
        logging.StreamHandler()
    ]
)
log = logging.getLogger(__name__)
log.info("Imports and logging configured successfully.")
log.info(f"Fraud data folder: {fraud_folder}")
log.info(f"Folder contents: {os.listdir(fraud_folder)}")

In [4]:
# ── CELL 4: Unzip Files ────────────────────────────────────────────────────────
zip_files = [
    "train_transaction.csv.zip",
    "train_identity.csv.zip",
    "archive (3).zip"
]

for zf in zip_files:
    zip_path = os.path.join(fraud_folder, zf)

    # Skip if the zip file is not found
    if not os.path.exists(zip_path):
        log.warning(f"{zf} not found in {fraud_folder} — skipping.")
        continue

    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            contents = z.namelist()
            # Print contents so you know exact filenames after extraction
            log.info(f"Extracting {zf} — contains: {contents}")
            z.extractall(fraud_folder)
            log.info(f"Successfully extracted {zf}")
    except zipfile.BadZipFile:
        # Catches corrupted or incomplete downloads
        log.error(f"{zf} is not a valid zip file. Try re-downloading from Kaggle.")
    except Exception as e:
        log.error(f"Unexpected error extracting {zf}: {e}")

# Show updated folder contents with file sizes to confirm extraction worked
log.info("Folder contents after extraction:")
for f in os.listdir(fraud_folder):
    size_mb = os.path.getsize(os.path.join(fraud_folder, f)) / (1024 * 1024)
    log.info(f"  {f}  ({size_mb:.1f} MB)")


In [5]:
# ── CELL 5: Load CSVs into DataFrames ─────────────────────────────────────────
# Only load columns your document builder uses.
# This skips 300+ V-columns in IEEE-CIS and cuts memory usage by ~80%,
# which prevents the Colab RAM crash on the free tier.
ieee_cols = [
    "TransactionID", "isFraud", "TransactionAmt", "TransactionDT",
    "ProductCD", "card1", "card2", "card3", "card4", "card5", "card6",
    "addr1", "addr2", "P_emaildomain", "R_emaildomain",
    "M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8", "M9"
]

identity_cols = [
    "TransactionID", "DeviceType", "DeviceInfo"
]

transaction_path = os.path.join(fraud_folder, "train_transaction.csv")
identity_path    = os.path.join(fraud_folder, "train_identity.csv")
paysim_path      = os.path.join(fraud_folder, "PS_20174392719_1491204439457_log.csv")

# Verify all three files exist before attempting to load
for path in [transaction_path, identity_path, paysim_path]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Expected file not found: {path}. "
            "Check the extraction output above for the correct filename."
        )
# Load IEEE-CIS transaction file with selected columns only
try:
    log.info("Loading train_transaction.csv with selected columns only...")
    ieee_trans = pd.read_csv(transaction_path, usecols=ieee_cols)
    log.info(f"Loaded: {ieee_trans.shape[0]:,} rows, {ieee_trans.shape[1]} columns")
except Exception as e:
    raise RuntimeError(f"Failed to load train_transaction.csv: {e}")

# Load IEEE-CIS identity file with selected columns only
try:
    log.info("Loading train_identity.csv with selected columns only...")
    ieee_identity = pd.read_csv(identity_path, usecols=identity_cols)
    log.info(f"Loaded: {ieee_identity.shape[0]:,} rows, {ieee_identity.shape[1]} columns")
except Exception as e:
    raise RuntimeError(f"Failed to load train_identity.csv: {e}")

# Load PaySim — smaller file so no column filtering needed
try:
    log.info("Loading PaySim CSV...")
    paysim_df = pd.read_csv(paysim_path, nrows=200000)  # limit to 200k rows to prevent RAM crash
    log.info(f"Loaded: {paysim_df.shape[0]:,} rows, {paysim_df.shape[1]} columns")
except Exception as e:
    raise RuntimeError(f"Failed to load PaySim CSV: {e}")

In [6]:
# ── CELL 6: Merge and Free Memory ─────────────────────────────────────────────
# Left join keeps all transactions even if they have no matching identity row.
# Those missing identity fields will be null in MongoDB, which is expected.
log.info("Merging IEEE-CIS transaction and identity files on TransactionID...")
try:
    ieee_df = ieee_trans.merge(ieee_identity, on="TransactionID", how="left")
    log.info(f"Merged shape: {ieee_df.shape[0]:,} rows, {ieee_df.shape[1]} columns")
    log.info(f"Transactions with identity info: "
             f"{ieee_identity.shape[0]:,} of {ieee_trans.shape[0]:,}")
except Exception as e:
    raise RuntimeError(f"Failed to merge IEEE-CIS files: {e}")

# Drop the unmerged frames immediately after merging to free RAM.
# On Colab free tier this is critical — without this the session will crash.
del ieee_trans
del ieee_identity
gc.collect()
log.info("Freed memory from unmerged IEEE-CIS frames.")

In [7]:
# ── CELL 7: Sanity Checks ──────────────────────────────────────────────────────
# Confirm isFraud label exists in both datasets before inserting anything
assert "isFraud" in ieee_df.columns,   "ERROR: isFraud missing from IEEE-CIS"
assert "isFraud" in paysim_df.columns, "ERROR: isFraud missing from PaySim"

# Print fraud rates — you will need these numbers for your rubric writeup
ieee_fraud_rate   = ieee_df["isFraud"].mean() * 100
paysim_fraud_rate = paysim_df["isFraud"].mean() * 100
total_docs        = len(ieee_df) + len(paysim_df)

log.info(f"IEEE-CIS fraud rate:       {ieee_fraud_rate:.2f}%")
log.info(f"PaySim fraud rate:         {paysim_fraud_rate:.2f}%")
log.info(f"Total documents to insert: {total_docs:,}")

# Warn early if we are below the rubric threshold
if total_docs < 1000:
    log.warning("Fewer than 1000 documents — rubric requires 1000+")
else:
    log.info("Rubric scale pre-check (1000+ documents): PASSED")

In [8]:
# ── CELL 8: Connect to MongoDB ─────────────────────────────────────────────────
try:
    mongodb = userdata.get("P-2")
    uri = (
        f"mongodb+srv://ega9cw_db_user:{mongodb}"
        f"@credit-card-fraud.cmgs8u1.mongodb.net/?appName=Credit-Card-Fraud"
    )
    client = MongoClient(uri, server_api=ServerApi('1'))

    # Ping to confirm the connection is live before inserting anything
    client.admin.command('ping')
    log.info("Successfully connected to MongoDB.")
except Exception as e:
    raise ConnectionError(f"Failed to connect to MongoDB: {e}")

# Point to your fraud database and transactions collection
db         = client["fraud_db"]
collection = db["transactions"]

# Log existing count so you know the state of the collection before insertion
existing = collection.count_documents({})
log.info(f"Documents already in collection before insertion: {existing:,}")

In [9]:
# ── CELL 9: Helper — Clean Invalid Float Values ────────────────────────────────
def clean_value(val):
    # MongoDB rejects Python's float('nan') and float('inf').
    # Convert them to None, which MongoDB stores as null.
    if isinstance(val, float) and (math.isnan(val) or math.isinf(val)):
        return None
    return val

In [10]:
# ── CELL 10: Document Builder — IEEE-CIS ──────────────────────────────────────
def build_ieee_document(row):
    # Converts one row of the merged IEEE-CIS dataframe into a nested
    # MongoDB document. Fields are grouped into sub-documents by category
    # to demonstrate the document model and satisfy the implicit schema
    # rubric requirement.
    return {
        # Source tag lets you filter by dataset in pipeline queries
        "source": "ieee_cis",

        # Top-level fraud label — the target variable for the model
        "is_fraud": bool(row["isFraud"]),

        # Core transaction fields available at the moment of approval
        "transaction": {
            "id":         row["TransactionID"],
            "amount":     clean_value(row["TransactionAmt"]),
            "dt":         clean_value(row["TransactionDT"]),  # timedelta not timestamp
            "product_cd": row.get("ProductCD")
        },

        # Card metadata — type and category are strong fraud signals
        "card": {
            "card1": clean_value(row.get("card1")),
            "card2": clean_value(row.get("card2")),
            "card3": clean_value(row.get("card3")),
            "card4": row.get("card4"),  # network e.g. visa/mastercard
            "card5": clean_value(row.get("card5")),
            "card6": row.get("card6")   # category e.g. debit/credit
        },

        # Billing address — mismatch between billing and shipping is a
        # known fraud signal
        "address": {
            "billing_region":  clean_value(row.get("addr1")),
            "billing_country": clean_value(row.get("addr2"))
        },

        # Email domains — certain free domains correlate with fraud
        "email": {
            "purchaser": row.get("P_emaildomain"),
            "recipient": row.get("R_emaildomain")
        },

        # Device info from identity file — null if no identity match exists
        "device": {
            "type": row.get("DeviceType"),
            "info": row.get("DeviceInfo")
        },

        # M1-M9 are Vesta's internal match flags — binary fraud indicators
        "match_flags": {
            f"M{i}": row.get(f"M{i}") for i in range(1, 10)
        }
    }

In [11]:
# ── CELL 11: Document Builder — PaySim ────────────────────────────────────────
def build_paysim_document(row):
    # Converts one row of the PaySim dataframe into a nested MongoDB document.
    # PaySim has account balance fields that IEEE-CIS lacks — this structural
    # difference between the two document types is an intentional demonstration
    # of implicit schema working as designed.
    return {
        # Source tag lets you filter by dataset in pipeline queries
        "source": "paysim",

        # Top-level fraud label — the target variable for the model
        "is_fraud": bool(row["isFraud"]),

        # Core transaction fields
        "transaction": {
            "amount": clean_value(row["amount"]),
            "type":   row["type"],     # CASH_OUT, PAYMENT, TRANSFER, etc.
            "step":   int(row["step"]) # 1 step = 1 hour of simulation time
        },

        # Account balances before and after — unique to PaySim.
        # A large drop in origin balance with no matching dest increase
        # is one of the strongest fraud signals in this dataset.
        "account": {
            "origin_balance_before": clean_value(row["oldbalanceOrg"]),
            "origin_balance_after":  clean_value(row["newbalanceOrig"]),
            "dest_balance_before":   clean_value(row["oldbalanceDest"]),
            "dest_balance_after":    clean_value(row["newbalanceDest"])
        }
    }

In [12]:
# ── CELL 12: Batch Insert Helper ──────────────────────────────────────────────
def insert_in_batches(df, build_fn, source_name, batch_size=1000):
    # Inserts documents in batches using bulk_write for efficiency.
    # ordered=False means one bad document will not stop the rest of
    # the batch from inserting.
    total_inserted = 0
    total_failed   = 0
    batch          = []

    for i, (_, row) in enumerate(df.iterrows()):
        try:
            # Build the document for this row and stage it in the batch
            doc = build_fn(row)
            batch.append(InsertOne(doc))
        except Exception as e:
            # Log the exact row so you can inspect it if needed
            log.warning(f"[{source_name}] Failed to build doc at row {i}: {e}")
            total_failed += 1
            continue

        # Once the batch is full, write it to MongoDB
        if len(batch) == batch_size:
            try:
                result = collection.bulk_write(batch, ordered=False)
                total_inserted += result.inserted_count
                log.info(
                    f"[{source_name}] Batch ending row {i} — "
                    f"running total: {total_inserted:,}"
                )
            except Exception as e:
                log.error(f"[{source_name}] Batch write failed at row {i}: {e}")
                total_failed += batch_size
            finally:
                # Always clear the batch whether it succeeded or failed
                batch = []

    # Insert any remaining documents that did not fill a complete batch
    if batch:
        try:
            result = collection.bulk_write(batch, ordered=False)
            total_inserted += result.inserted_count
            log.info(
                f"[{source_name}] Final batch inserted — "
                f"running total: {total_inserted:,}"
            )
        except Exception as e:
            log.error(f"[{source_name}] Final batch write failed: {e}")
            total_failed += len(batch)

    log.info(
        f"[{source_name}] Complete — "
        f"inserted: {total_inserted:,}, failed: {total_failed:,}"
    )
    return total_inserted

In [13]:
# ── CELL 13: Run Insertions ────────────────────────────────────────────────────
log.info("Starting IEEE-CIS insertion...")
ieee_inserted = insert_in_batches(ieee_df, build_ieee_document, "IEEE-CIS")

log.info("Starting PaySim insertion...")
paysim_inserted = insert_in_batches(paysim_df, build_paysim_document, "PaySim")

In [14]:
# ── CELL 14: Final Verification ───────────────────────────────────────────────
# Query MongoDB directly to confirm counts — do not rely on the insertion
# counters alone since bulk_write can silently skip duplicates
total_in_db  = collection.count_documents({})
ieee_in_db   = collection.count_documents({"source": "ieee_cis"})
paysim_in_db = collection.count_documents({"source": "paysim"})
fraud_in_db  = collection.count_documents({"is_fraud": True})
legit_in_db  = collection.count_documents({"is_fraud": False})

log.info("── Final Verification ──────────────────────────────────────")
log.info(f"Total documents in collection:   {total_in_db:,}")
log.info(f"  From IEEE-CIS:                 {ieee_in_db:,}")
log.info(f"  From PaySim:                   {paysim_in_db:,}")
log.info(f"  Fraudulent transactions:       {fraud_in_db:,}")
log.info(f"  Legitimate transactions:       {legit_in_db:,}")
log.info(f"  Overall fraud rate in DB:      {fraud_in_db/total_in_db*100:.2f}%")

# Rubric scale checks
if total_in_db >= 1000:
    log.info("Scale: 1000+ documents — PASSED")
if total_in_db >= 100:
    log.info("Scale: 100+ documents  — PASSED")
if total_in_db >= 10:
    log.info("Scale: 10+ documents   — PASSED")

if total_in_db < 1000:
    log.warning(
        f"Rubric requirement NOT met — only {total_in_db:,} documents. "
        "Re-run with a larger nrows value."
    )

In [15]:
# Explore the DB
import pprint

log.info("Running sanity checks on inserted data...")

# ── 1. Collection Overview ─────────────────────────────────────────────────────
print("\n" + "="*60)
print("COLLECTION OVERVIEW")
print("="*60)
total    = collection.count_documents({})
ieee_ct  = collection.count_documents({"source": "ieee_cis"})
pay_ct   = collection.count_documents({"source": "paysim"})
fraud_ct = collection.count_documents({"is_fraud": True})
legit_ct = collection.count_documents({"is_fraud": False})

print(f"Total documents:       {total:,}")
print(f"  IEEE-CIS:            {ieee_ct:,}")
print(f"  PaySim:              {pay_ct:,}")
print(f"  Fraudulent:          {fraud_ct:,}")
print(f"  Legitimate:          {legit_ct:,}")
print(f"  Fraud rate:          {fraud_ct/total*100:.2f}%")


# ── 2. Sample IEEE-CIS Document ────────────────────────────────────────────────
print("\n" + "="*60)
print("SAMPLE IEEE-CIS DOCUMENT")
print("="*60)
ieee_sample = collection.find_one({"source": "ieee_cis"})
pprint.pprint(ieee_sample)


# ── 3. Sample PaySim Document ──────────────────────────────────────────────────
print("\n" + "="*60)
print("SAMPLE PAYSIM DOCUMENT")
print("="*60)
pay_sample = collection.find_one({"source": "paysim"})
pprint.pprint(pay_sample)


# ── 4. Sample Fraudulent Document from Each Source ────────────────────────────
print("\n" + "="*60)
print("SAMPLE FRAUDULENT IEEE-CIS DOCUMENT")
print("="*60)
fraud_ieee = collection.find_one({"source": "ieee_cis", "is_fraud": True})
pprint.pprint(fraud_ieee)

print("\n" + "="*60)
print("SAMPLE FRAUDULENT PAYSIM DOCUMENT")
print("="*60)
fraud_pay = collection.find_one({"source": "paysim", "is_fraud": True})
pprint.pprint(fraud_pay)


# ── 5. Implicit Schema Check ───────────────────────────────────────────────────
# Confirms that IEEE-CIS docs have 'device' and PaySim docs have 'account',
# proving the two document types have different structures as intended
print("\n" + "="*60)
print("IMPLICIT SCHEMA CHECK")
print("="*60)

ieee_with_device  = collection.count_documents({"source": "ieee_cis", "device": {"$exists": True}})
pay_with_account  = collection.count_documents({"source": "paysim",   "account": {"$exists": True}})
ieee_with_account = collection.count_documents({"source": "ieee_cis", "account": {"$exists": True}})
pay_with_device   = collection.count_documents({"source": "paysim",   "device":  {"$exists": True}})

print(f"IEEE-CIS docs with 'device' sub-doc:   {ieee_with_device:,}  (should equal {ieee_ct:,})")
print(f"PaySim docs with 'account' sub-doc:    {pay_with_account:,}  (should equal {pay_ct:,})")
print(f"IEEE-CIS docs with 'account' sub-doc:  {ieee_with_account:,}  (should be 0)")
print(f"PaySim docs with 'device' sub-doc:     {pay_with_device:,}   (should be 0)")


# ── 6. Top 5 Transaction Types in PaySim ──────────────────────────────────────
print("\n" + "="*60)
print("PAYSIM TRANSACTION TYPE BREAKDOWN")
print("="*60)
pipeline = [
    {"$match":  {"source": "paysim"}},
    {"$group":  {"_id": "$transaction.type", "count": {"$sum": 1}}},
    {"$sort":   {"count": -1}}
]
for doc in collection.aggregate(pipeline):
    print(f"  {doc['_id']:<12} {doc['count']:,}")


# ── 7. Fraud by Transaction Type in PaySim ────────────────────────────────────
print("\n" + "="*60)
print("PAYSIM FRAUD BY TRANSACTION TYPE")
print("="*60)
pipeline = [
    {"$match":  {"source": "paysim", "is_fraud": True}},
    {"$group":  {"_id": "$transaction.type", "fraud_count": {"$sum": 1}}},
    {"$sort":   {"fraud_count": -1}}
]
for doc in collection.aggregate(pipeline):
    print(f"  {doc['_id']:<12} {doc['fraud_count']:,} fraudulent transactions")


# ── 8. Average Transaction Amount: Fraud vs Legitimate ────────────────────────
print("\n" + "="*60)
print("AVERAGE TRANSACTION AMOUNT: FRAUD VS LEGITIMATE")
print("="*60)
for source in ["ieee_cis", "paysim"]:
    amount_field = "transaction.amount"
    pipeline = [
        {"$match":  {"source": source, "transaction.amount": {"$ne": None}}},
        {"$group":  {
            "_id":        "$is_fraud",
            "avg_amount": {"$avg": f"${amount_field}"}
        }},
        {"$sort": {"_id": 1}}
    ]
    print(f"\n  {source.upper()}")
    for doc in collection.aggregate(pipeline):
        label = "Fraud" if doc["_id"] else "Legitimate"
        print(f"    {label:<12} avg amount: ${doc['avg_amount']:,.2f}")


# ── 9. Null Check on Key Fields ────────────────────────────────────────────────
# Confirms expected nulls (device info on IEEE-CIS where no identity matched)
# and flags unexpected nulls that might indicate a data loading issue
print("\n" + "="*60)
print("NULL CHECK ON KEY FIELDS")
print("="*60)
null_checks = [
    ("IEEE-CIS missing transaction.amount", {"source": "ieee_cis", "transaction.amount": None}),
    ("IEEE-CIS missing card.card4",         {"source": "ieee_cis", "card.card4": None}),
    ("IEEE-CIS missing device.type",        {"source": "ieee_cis", "device.type": None}),
    ("PaySim missing transaction.amount",   {"source": "paysim",   "transaction.amount": None}),
    ("PaySim missing account.origin_balance_before", {"source": "paysim", "account.origin_balance_before": None}),
]
for label, query in null_checks:
    count = collection.count_documents(query)
    flag  = "  ← expected" if "device" in label else ("  ← CHECK THIS" if count > 0 else "")
    print(f"  {label:<45} {count:,}{flag}")


print("\n" + "="*60)
print("SANITY CHECK COMPLETE")
print("="*60)
log.info("Sanity check complete.")


COLLECTION OVERVIEW
Total documents:       823,540
  IEEE-CIS:            623,540
  PaySim:              200,000
  Fraudulent:          21,905
  Legitimate:          801,635
  Fraud rate:          2.66%

SAMPLE IEEE-CIS DOCUMENT
{'_id': ObjectId('69e27938021fa7f73d7841e0'),
 'address': {'billing_country': 87.0, 'billing_region': 315.0},
 'card': {'card1': 4806,
          'card2': 490.0,
          'card3': 150.0,
          'card4': 'visa',
          'card5': 226.0,
          'card6': 'debit'},
 'device': {'info': nan, 'type': nan},
 'email': {'purchaser': 'gmail.com', 'recipient': nan},
 'is_fraud': False,
 'match_flags': {'M1': nan,
                 'M2': nan,
                 'M3': nan,
                 'M4': 'M0',
                 'M5': 'T',
                 'M6': 'T',
                 'M7': nan,
                 'M8': nan,
                 'M9': nan},
 'source': 'ieee_cis',
 'transaction': {'amount': 29.0,
                 'dt': 8023328,
                 'id': 3312000,
            